# 1. Logic Initialization


In [1]:
import pandas as pd
import pathlib

RAW_DATA_PATH = pathlib.Path("data/raw")
PROCESSED_DATA_PATH = pathlib.Path("data/processed")
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

csv_files = sorted(list(RAW_DATA_PATH.glob("*.csv")))

# 2. Robust Parsing Functions
The SJPD dataset has inconsistent timestamp formats across years (e.g., some have 'PS' suffixes, others use different delimiters). We define custom functions to handle this complexity.

In [2]:
def parse_cdts_series(raw_series: pd.Series) -> pd.Series:
    # Normalize raw values and remove known suffix noise.
    s = (
        raw_series.astype("string")
        .str.upper()
        .str.replace("PS", "", regex=False)
        .str.strip()
    )

    # Primary parse: first 14 digits interpreted as YYYYMMDDHHMMSS.
    digits14 = s.str.extract(r"(\d{14})", expand=False)
    parsed = pd.to_datetime(digits14, format="%Y%m%d%H%M%S", errors="coerce")

    # Secondary parse: 12-digit timestamps (YYYYMMDDHHMM) -> append seconds.
    missing_mask = parsed.isna()
    if missing_mask.any():
        digits12 = s[missing_mask].str.extract(r"(\d{12})", expand=False)
        parsed12 = pd.to_datetime(
            digits12 + "00", format="%Y%m%d%H%M%S", errors="coerce"
        )
        parsed.loc[missing_mask] = parsed12.values

    # Final fallback: let pandas infer any remaining odd formats.
    missing_mask = parsed.isna()
    if missing_mask.any():
        parsed_fallback = pd.to_datetime(s[missing_mask], errors="coerce")
        parsed.loc[missing_mask] = parsed_fallback.values

    return parsed


def check_data_quality(df: pd.DataFrame, file_name: str):
    """Checks for and prints problematic rows in the dataframe."""
    print(f"--- Data Quality Audit for {file_name} ---")

    # Check for missing timestamps
    missing_cdts = df["CDTS"].isna().sum()
    if missing_cdts > 0:
        print(f"[WARNING] Found {missing_cdts} rows with missing CDTS (timestamps).")

    # Check for missing priority
    if "PRIORITY" in df.columns:
        missing_priority = df["PRIORITY"].isna().sum()
        if missing_priority > 0:
            print(f"[WARNING] Found {missing_priority} rows with missing PRIORITY.")
    else:
        print(f"[ERROR] 'PRIORITY' column missing entirely!")

    # Sample problematic rows
    problematic = df[
        df["CDTS"].isna()
        | (df["PRIORITY"].isna() if "PRIORITY" in df.columns else False)
    ]
    if not problematic.empty:
        print("Sample problematic rows:")
        # Use print head instead of display for simplicity in script-based check if needed,
        # but for notebook it's fine to keep 'display' if the environment supports it.
        print(problematic.head(3))
    else:
        print("No immediate data quality issues detected.")
    print("------------------------------------------\n")

# 3. Sequential File Processing
We load each file, apply our cleaning logic, and flag important features like 'Canceled' calls and 'Priority 1' calls.

In [3]:
processed_dfs = []

for file_path in csv_files:
    print(f"Processing {file_path.name}...")
    df = pd.read_csv(file_path, low_memory=False)
    check_data_quality(df, file_path.name)

    # Capture raw CDTS for traceability checks
    df["CDTS_RAW"] = df["CDTS"].astype("string")

    # Apply robust parsing
    df["CDTS"] = parse_cdts_series(df["CDTS"])

    # Simple flags for outcome tracking
    if "FINAL_DISPO" in df.columns:
        df["is_canceled"] = df["FINAL_DISPO"].str.contains("CAN", case=False, na=False)
    else:
        df["is_canceled"] = False

    if "PRIORITY" in df.columns:
        df["is_priority_1"] = df["PRIORITY"].astype(str).str.contains("1", na=False)
    else:
        df["is_priority_1"] = False

    processed_dfs.append(df)

master_df = pd.concat(processed_dfs, ignore_index=True)

# Add year column for easy trend analysis and filtering
master_df["year"] = master_df["CDTS"].dt.year.astype("int16")

Processing policecalls2016.csv...


--- Data Quality Audit for policecalls2016.csv ---
No immediate data quality issues detected.
------------------------------------------



Processing policecalls2017.csv...


--- Data Quality Audit for policecalls2017.csv ---
No immediate data quality issues detected.
------------------------------------------



Processing policecalls2018.csv...


...

--- Data Quality Audit for policecalls2024.csv ---
No immediate data quality issues detected.
------------------------------------------



Processing policecalls2025.csv...


--- Data Quality Audit for policecalls2025.csv ---
No immediate data quality issues detected.
------------------------------------------



Processing policecalls2026.csv...


--- Data Quality Audit for policecalls2026.csv ---
No immediate data quality issues detected.
------------------------------------------



# 4. Quality & Traceability Check
Let's compare the raw input vs the cleaned output to ensure our parsing logic didn't drop data.

In [4]:
print("=== Transformation: Before vs After ===")
comparison_cols = [
    "CDTS_RAW",
    "CDTS",
    "FINAL_DISPO",
    "is_canceled",
    "PRIORITY",
    "is_priority_1",
]
# Select columns that exist in the dataframe
available_cols = [c for c in comparison_cols if c in master_df.columns]
display(master_df[available_cols].head(10))

print(f"\nTotal records processed: {len(master_df):,}")
print(f"Missing timestamps after parsing: {master_df['CDTS'].isna().sum()}")

=== Transformation: Before vs After ===


,CDTS_RAW,CDTS,FINAL_DISPO,is_canceled,PRIORITY,is_priority_1
0,20160514222222PD,2016-05-14 22:22:22,No report required; dispatch r,False,3,False
1,20160514214406PD,2016-05-14 21:44:06,No Response,False,4,False
2,20160514212618PD,2016-05-14 21:26:18,No Response,False,3,False
3,20160514225346PD,2016-05-14 22:53:46,No Response,False,4,False
4,20160514224627PD,2016-05-14 22:46:27,Report taken,False,2,False
5,20160514233025PD,2016-05-14 23:30:25,Canceled,True,4,False
6,20160514215732PD,2016-05-14 21:57:32,No report required; dispatch r,False,4,False
7,20160514223058PD,2016-05-14 22:30:58,No report required; dispatch r,False,2,False
8,20160514225108PD,2016-05-14 22:51:08,Turned over To (TOT),False,2,False
9,20160514223844PD,2016-05-14 22:38:44,Gone on Arrival/unable to loca,False,4,False



Total records processed: 3,077,538
Missing timestamps after parsing: 0


# 5. Parquet Data Export
We save the combined file to the `adjusted` directory for use in downstream EDA and Modeling notebooks.

In [5]:
min_year = master_df["year"].min()
max_year = master_df["year"].max()
out_path = PROCESSED_DATA_PATH / f"sjpd_calls_{min_year}_{max_year}_processed.parquet"
master_df.to_parquet(out_path, index=False)
print(f"Saved cleaned dataset to: {out_path}")

Saved cleaned dataset to: data\processed\sjpd_calls_2016_2026_processed.parquet
